# 04 · Evaluación M1 vs M2 v2 sobre `val` (500)

Compara, documento por documento, lo que extrae cada método con la anotación de DocILE.

| Entrada | Origen |
|---|---|
| Anotaciones (referencia) | `/Volumes/workspace/invoice_extraction_benchmark/eval_inputs/annotations_val/` (500 JSON de DocILE, licencia solo investigación) |
| M1 · LayoutLMv3 | `/Volumes/workspace/invoice_extraction_benchmark/eval_inputs/m1_val_schema.json` (predicciones publicadas, adaptadas con `tools/m1_to_schema.py`) |
| M2 v2 · ai_parse_document + ai_extract | tabla `workspace.invoice_extraction_benchmark.m2v2_extracted_val` (notebook 03, celdas 3 y 4) |
| Grupo de diseño (0-shot, 1-3, 4+) | `/Volumes/workspace/invoice_extraction_benchmark/eval_inputs/val_grupos.json` |

**Métrica por valor:** cantidad exacta, costo unitario |Δ| < 0.01, importe |Δ| < 0.50 (código en `tools/eval_valores.py`).
**Pruebas:** Shapiro-Wilk → Wilcoxon o t pareada (HE2, HE3), McNemar exacto por campo (HE1), IC 95 % bootstrap (`tools/comparar.py`).

Salida: tablas `ieb_eval_resumen`, `ieb_eval_por_doc` y `ieb_eval_detalle` (esta última con texto de facturas: privada).

In [ ]:
# Celda 1: código de evaluación del repositorio
import json, os, sys
sys.path.insert(0, os.path.abspath("../tools"))
from eval_valores import HEADER
from comparar import comparar, doc_score

In [ ]:
# Celda 2: entradas
V = "/Volumes/workspace/invoice_extraction_benchmark/eval_inputs"
m1 = json.load(open(f"{V}/m1_val_schema.json", encoding="utf-8"))
G = {d["docid"]: d["grupo"] for d in json.load(open(f"{V}/val_grupos.json", encoding="utf-8"))}
m2 = {r.docid: json.loads(r.j) for r in spark.sql(
    "SELECT docid, to_json(extracted) AS j FROM workspace.invoice_extraction_benchmark.m2v2_extracted_val").collect()}
print(f"docs: grupos={len(G)}  M1={len(m1)}  M2={len(m2)}")
assert set(G) <= set(m1) and set(G) <= set(m2)

In [ ]:
# Celda 3: evaluar y probar hipótesis
res, R1, R2 = comparar(f"{V}/annotations_val", m1, m2, G)
print(json.dumps(res, ensure_ascii=False, indent=1, default=lambda x: round(x, 4) if isinstance(x, float) else str(x)))

In [ ]:
# Celda 4: resumen (cifras agregadas, publicable)
r4 = lambda x: None if x is None else round(float(x), 4)
rows = []
for k in ("cantidad", "costo_unitario", "importe", "completa"):
    rows.append(("Líneas emparejadas", k, r4(res["agregado_M1"][k]), r4(res["agregado_M2"][k]), None, None, None, None, None))
for k in ("lineas_anotadas", "lineas_extraidas", "emparejadas"):
    rows.append(("Líneas (conteo)", k, res["agregado_M1"][k], res["agregado_M2"][k], None, None, None, None, None))
for g, t in res["lineas"].items():
    rows.append(("HE2/HE3 líneas completas por documento", g, r4(t["media_M1"]), r4(t["media_M2"]), t["n"],
                 t["prueba"], r4(t["p"]), r4(t["ic95_dif"][0]), r4(t["ic95_dif"][1])))
for k, t in res["cabecera"].items():
    rows.append(("HE1 cabecera", k, r4(t["M1"]), r4(t["M2"]), t["n"], "McNemar exacto", r4(t["p"]), None, None))
for nm in ("M1", "M2"):
    a = res["alertas"][nm]
    rows.append(("HE5 alertas", f"{nm}: señaladas / con error (falsas alarmas {a['falsas_alarmas']})",
                 a["señaladas"] if nm == "M1" else None, a["señaladas"] if nm == "M2" else None,
                 a["facturas_con_error"], None, None, None, None))
cols = "seccion indicador M1 M2 n prueba p ic95_inf ic95_sup".split()
resumen = spark.createDataFrame([dict(zip(cols, r)) for r in rows],
    "seccion string, indicador string, M1 double, M2 double, n int, prueba string, p double, ic95_inf double, ic95_sup double")
resumen.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.invoice_extraction_benchmark.ieb_eval_resumen")
display(resumen)

In [ ]:
# Celda 5: por documento (solo indicadores, sin texto de facturas, publicable)
pd_rows = []
for d in sorted(G):
    a, b = R1[d], R2[d]
    pd_rows.append({"docid": d, "grupo": G[d], "lineas_anotadas": a["n_ref_lines"],
        "M1_lineas_extraidas": a["n_pred_lines"], "M2_lineas_extraidas": b["n_pred_lines"],
        "M1_lineas_completas": r4(doc_score(a)), "M2_lineas_completas": r4(doc_score(b)),
        "M1_cabecera_ok": sum(h["ok"] for h in a["head"].values()), "M2_cabecera_ok": sum(h["ok"] for h in b["head"].values()),
        "cabecera_anotada": sum(h["ref"] for h in a["head"].values()),
        "M1_error": a["error"], "M1_alerta": a["alerta"], "M2_error": b["error"], "M2_alerta": b["alerta"]})
por_doc = spark.createDataFrame(pd_rows,
    "docid string, grupo string, lineas_anotadas int, M1_lineas_extraidas int, M2_lineas_extraidas int, "
    "M1_lineas_completas double, M2_lineas_completas double, M1_cabecera_ok int, M2_cabecera_ok int, "
    "cabecera_anotada int, M1_error boolean, M1_alerta boolean, M2_error boolean, M2_alerta boolean")
por_doc.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.invoice_extraction_benchmark.ieb_eval_por_doc")
display(por_doc)

In [ ]:
# Celda 6: detalle valor por valor (CONTIENE texto de facturas: solo hoja restringida)
s = lambda x: None if x is None else str(x)
ok = lambda x: None if x is None else ("sí" if x else "no")
det = []
for nm, R in (("M1", R1), ("M2", R2)):
    for d in sorted(G):
        for k, h in R[d]["head"].items():
            if h["ref"] or h["pred"]:
                det.append((nm, d, G[d], "cabecera", k, s(h["ref_txt"]), s(h["pred_txt"]), ok(h["ok"]) if h["ref"] else "sin anotación"))
        for x in R[d]["detail"]:
            a, b = x["ref"] or {}, x["pred"] or {}
            for k in ("descripcion", "cantidad", "costo_unitario", "importe"):
                det.append((nm, d, G[d], "linea: " + x["estado"], k, s(a.get(k)), s(b.get(k)),
                            ok(x.get(k)) if k != "descripcion" else None))
detalle = spark.createDataFrame(det, "modelo string, docid string, grupo string, tipo string, campo string, anotado string, extraido string, acierto string")
detalle.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.invoice_extraction_benchmark.ieb_eval_detalle")
print(detalle.count(), "filas de detalle")